# Día 3 · Estimación puntual y el primer intervalo de confianza

**Apartados del documento que cubre este día:** 4 (Estimación puntual) y la primera parte del 5 (Estimación por intervalo): el intervalo para una proporción.

**Al terminar este día, cada integrante del equipo debe poder explicar sin leer:**
1. Qué es una estimación puntual y cómo se interpreta (el documento dice que la interpretación es obligatoria).
2. Por qué una estimación puntual no basta.
3. Qué es el error estándar.
4. De dónde sale el número 1.96.
5. Cómo se construye el intervalo de confianza para P1, número por número.
6. Cómo se interpreta ese intervalo sin cometer el error más común.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

RAIZ = Path.cwd().parent if Path.cwd().name == "clases" else Path.cwd()
poblacion = pd.read_csv(RAIZ / "data" / "processed" / "poblacion_clientes.csv")
muestra = pd.read_csv(RAIZ / "data" / "processed" / "muestra_clientes_n1000.csv")
N = len(poblacion)
n = len(muestra)
print("Clientes en la población (N):", N)
print("Clientes en la muestra (n):", n)

Clientes en la población (N): 10000
Clientes en la muestra (n): 1000


---
## Paso 1 · Qué pide el documento

**Apartado 4 · Estimación puntual.** Para cada pregunta hay que identificar:
- El **parámetro**: la característica de la población que se quiere conocer (por ejemplo, μ).
- El **estadístico**: el valor calculado con la muestra (por ejemplo, x̄).
- La **interpretación**. El documento lo dice textualmente: no basta escribir "la media es 7.8". Hay que explicar que ese valor es una estimación del valor de la población.

**Apartado 5 · Estimación por intervalo.** Al menos dos intervalos de confianza. El proyecto tiene cuatro: hoy se construye el primero (P1, una proporción) y en el Día 4 los otros tres.

---
## Paso 2 · Las cuatro estimaciones puntuales

Repaso del Día 1: el **parámetro** se calcula con toda la población y no se conoce; el **estadístico** se calcula con la muestra y se usa para estimarlo. Una **estimación puntual** es usar **un solo número** de la muestra como estimación del parámetro.

| Pregunta | Parámetro (población) | Estadístico (muestra) |
|---|---|---|
| P1 | $p$ = proporción de clientes que abandonan | $\hat p$ ("p gorro") |
| P2 | $\mu$ = saldo promedio de quienes abandonan | $\bar x$ ("x barra") |
| P3 | $p_1 - p_2$ = proporción de abandono en inactivos menos en activos | $\hat p_1 - \hat p_2$ |
| P4 | $\mu_1 - \mu_2$ = edad promedio de quienes se van menos la de quienes se quedan | $\bar x_1 - \bar x_2$ |

In [2]:
p_gorro = muestra["abandono"].mean()
print("P1 · Proporción de abandono en la muestra (p gorro):", round(p_gorro, 4))

P1 · Proporción de abandono en la muestra (p gorro): 0.206


In [3]:
saldo_abandona = muestra.loc[muestra["abandono"] == 1, "saldo"]
media_saldo_abandona = saldo_abandona.mean()
print("P2 · Saldo promedio de quienes abandonaron (x barra):", round(media_saldo_abandona, 2))

P2 · Saldo promedio de quienes abandonaron (x barra): 91561.53


In [4]:
abandono_inactivos = muestra.loc[muestra["miembro_activo"] == 0, "abandono"]
abandono_activos = muestra.loc[muestra["miembro_activo"] == 1, "abandono"]
p_gorro_inactivos = abandono_inactivos.mean()
p_gorro_activos = abandono_activos.mean()
diferencia_proporciones = p_gorro_inactivos - p_gorro_activos
print("P3 · Proporción de abandono en inactivos:", round(p_gorro_inactivos, 4))
print("P3 · Proporción de abandono en activos:", round(p_gorro_activos, 4))
print("P3 · Diferencia (inactivos - activos):", round(diferencia_proporciones, 4))

P3 · Proporción de abandono en inactivos: 0.2657
P3 · Proporción de abandono en activos: 0.1546
P3 · Diferencia (inactivos - activos): 0.1111


In [5]:
edad_se_va = muestra.loc[muestra["abandono"] == 1, "edad"]
edad_se_queda = muestra.loc[muestra["abandono"] == 0, "edad"]
media_edad_se_va = edad_se_va.mean()
media_edad_se_queda = edad_se_queda.mean()
diferencia_medias_edad = media_edad_se_va - media_edad_se_queda
print("P4 · Edad promedio de quienes se fueron:", round(media_edad_se_va, 2))
print("P4 · Edad promedio de quienes se quedaron:", round(media_edad_se_queda, 2))
print("P4 · Diferencia (se fue - se quedó):", round(diferencia_medias_edad, 2))

P4 · Edad promedio de quienes se fueron: 46.5
P4 · Edad promedio de quienes se quedaron: 37.1
P4 · Diferencia (se fue - se quedó): 9.41


**Comprobación:** los tamaños de cada grupo.

In [6]:
print("Clientes que abandonaron:", len(saldo_abandona))
print("Clientes inactivos:", len(abandono_inactivos))
print("Clientes activos:", len(abandono_activos))
print("Clientes que se quedaron:", len(edad_se_queda))

Clientes que abandonaron: 206
Clientes inactivos: 463
Clientes activos: 537
Clientes que se quedaron: 794


---
## Paso 3 · La interpretación obligatoria

El documento da este modelo: *"La media muestral obtenida fue de 7.8 minutos. Este valor constituye una estimación puntual del tiempo promedio poblacional."*

Aplicado al proyecto, así se escribe cada una:

- **P1.** La proporción muestral de abandono fue de **0.206** (206 de 1,000 clientes). Este valor es una estimación puntual de la proporción de abandono de los 10,000 clientes del banco: se estima que cerca de 2,060 clientes abandonaron.
- **P2.** El saldo promedio muestral de los clientes que abandonaron fue de **91,562 €**. Este valor es una estimación puntual del saldo promedio de todos los clientes del banco que abandonan.
- **P3.** La diferencia muestral entre la proporción de abandono de inactivos (0.266) y activos (0.155) fue de **0.111**, es decir, 11.1 puntos porcentuales. Es una estimación puntual de esa diferencia en los 10,000 clientes.
- **P4.** La diferencia muestral entre la edad promedio de quienes abandonaron (46.5) y quienes se quedaron (37.1) fue de **9.41 años**. Es una estimación puntual de esa diferencia en los 10,000 clientes.

**Lo que NO se debe escribir:** "el abandono es 0.206". Ese número es de la muestra; el de la población no se conoce.

---
## Paso 4 · Por qué un solo número no basta

Si se hubieran sorteado otros 1,000 clientes, la proporción habría salido distinta. Se sacan cinco muestras con cinco semillas diferentes:

In [7]:
p_muestra_semilla_1 = poblacion["abandono"].sample(n, random_state=1).mean()
p_muestra_semilla_2 = poblacion["abandono"].sample(n, random_state=2).mean()
p_muestra_semilla_3 = poblacion["abandono"].sample(n, random_state=3).mean()
p_muestra_semilla_4 = poblacion["abandono"].sample(n, random_state=4).mean()
p_muestra_semilla_5 = poblacion["abandono"].sample(n, random_state=5).mean()
print("Proporción con la muestra 1:", p_muestra_semilla_1)
print("Proporción con la muestra 2:", p_muestra_semilla_2)
print("Proporción con la muestra 3:", p_muestra_semilla_3)
print("Proporción con la muestra 4:", p_muestra_semilla_4)
print("Proporción con la muestra 5:", p_muestra_semilla_5)

Proporción con la muestra 1: 0.211
Proporción con la muestra 2: 0.188
Proporción con la muestra 3: 0.197
Proporción con la muestra 4: 0.197
Proporción con la muestra 5: 0.206


Cada muestra da un valor diferente. La estimación puntual no dice **cuánto podría estar alejada** del valor real. Para eso se construye un **intervalo de confianza**: un rango de valores, calculado con la muestra, dentro del cual se espera que esté el parámetro.

---
## Paso 5 · El error estándar

El **error estándar** mide cuánto cambia la estimación de una muestra a otra. Para una proporción:

$$EE = \sqrt{\frac{\hat p\,(1 - \hat p)}{n}}$$

| Símbolo | Qué es | Valor |
|---|---|---|
| $\hat p$ | Proporción de la muestra | 0.206 |
| $1 - \hat p$ | Proporción que no abandonó | 0.794 |
| $n$ | Tamaño de la muestra | 1,000 |

Observa que $n$ está dividiendo: **con más clientes, el error estándar es más pequeño**.

In [8]:
producto_p = p_gorro * (1 - p_gorro)
cociente = producto_p / n
error_estandar = np.sqrt(cociente)
print("p gorro por (1 - p gorro):", round(producto_p, 6))
print("Dividido entre n:", round(cociente, 8))
print("Error estándar (raíz cuadrada):", round(error_estandar, 4))

p gorro por (1 - p gorro): 0.163564
Dividido entre n: 0.00016356
Error estándar (raíz cuadrada): 0.0128


**Lectura:** el error estándar es 0.0128, es decir, 1.28 puntos porcentuales. Es el tamaño típico de la variación de la proporción de una muestra a otra.

**Comprobación:** en el Paso 4, las cinco muestras dieron valores entre 0.188 y 0.211. Están a una distancia de 1 o 2 errores estándar de 0.204.

---
## Paso 6 · De dónde sale el 1.96

Si se sacaran muchísimas muestras, sus proporciones formarían una curva en forma de campana: la **distribución normal**. En una distribución normal se cumple una regla fija:

**el 95 % de los valores está a menos de 1.96 errores estándar del centro.**

El número 1.96 se llama **valor crítico** y se escribe $z_{\alpha/2}$:
- $\alpha$ (alfa) es lo que se acepta fallar: 100 % − 95 % = 5 % = 0.05.
- Se reparte en los dos lados de la campana: 0.025 a la izquierda y 0.025 a la derecha. Por eso se escribe $\alpha/2$.

Se comprueba con Python:

In [9]:
nivel_confianza = 0.95
alfa = 1 - nivel_confianza
z_critico = stats.norm.ppf(1 - alfa / 2)
area_entre_menos_y_mas_z = stats.norm.cdf(z_critico) - stats.norm.cdf(-z_critico)
print("Alfa:", round(alfa, 2))
print("Alfa entre 2:", round(alfa / 2, 3))
print("Valor crítico z:", round(z_critico, 4))
print("Área de la campana entre -1.96 y +1.96:", round(area_entre_menos_y_mas_z, 4))

Alfa: 0.05
Alfa entre 2: 0.025
Valor crítico z: 1.96
Área de la campana entre -1.96 y +1.96: 0.95


---
## Paso 7 · El margen de error y el intervalo sin corrección

La fórmula del intervalo para una proporción es la que aparece en el documento:

$$IC = \hat p \pm z_{\alpha/2}\sqrt{\frac{\hat p(1-\hat p)}{n}}$$

Se calcula en dos partes:
1. **Margen de error** = valor crítico × error estándar.
2. **Intervalo** = estimación − margen hasta estimación + margen.

In [10]:
margen_error_sin_correccion = z_critico * error_estandar
limite_inferior_sin_correccion = p_gorro - margen_error_sin_correccion
limite_superior_sin_correccion = p_gorro + margen_error_sin_correccion
print("Margen de error:", round(margen_error_sin_correccion, 4))
print("Límite inferior:", round(limite_inferior_sin_correccion, 4))
print("Límite superior:", round(limite_superior_sin_correccion, 4))

Margen de error: 0.0251
Límite inferior: 0.1809
Límite superior: 0.2311


---
## Paso 8 · La corrección por población finita

La fórmula anterior supone una población muy grande. Nuestra muestra es el **10 %** de la población. Cuando la muestra es más del 5 % de la población, el error estándar se multiplica por el **factor de corrección por población finita (FCPF)**:

$$\text{FCPF} = \sqrt{\frac{N - n}{N - 1}}$$

**Razón:** si se revisa una parte grande de la población, queda menos por conocer y la estimación varía menos de una muestra a otra. Si se revisara a toda la población ($n = N$), el factor valdría 0 y no habría error.

In [11]:
fcpf = np.sqrt((N - n) / (N - 1))
error_estandar_corregido = error_estandar * fcpf
margen_error = z_critico * error_estandar_corregido
limite_inferior = p_gorro - margen_error
limite_superior = p_gorro + margen_error
print("FCPF:", round(fcpf, 4))
print("Error estándar corregido:", round(error_estandar_corregido, 4))
print("Margen de error corregido:", round(margen_error, 4))
print("Intervalo de confianza al 95 % para p:", (float(round(limite_inferior, 4)), float(round(limite_superior, 4))))

FCPF: 0.9487
Error estándar corregido: 0.0121
Margen de error corregido: 0.0238
Intervalo de confianza al 95 % para p: (0.1822, 0.2298)


**Comprobación:** scipy calcula el mismo intervalo con `stats.norm.interval`, dándole el centro y el error estándar corregido.

In [12]:
intervalo_scipy = stats.norm.interval(0.95, loc=p_gorro, scale=error_estandar_corregido)
print("Intervalo calculado con scipy:", (float(round(intervalo_scipy[0], 4)), float(round(intervalo_scipy[1], 4))))

Intervalo calculado con scipy: (0.1822, 0.2298)


**Condición de uso:** la fórmula funciona si hay al menos 10 clientes que abandonan y 10 que no.

In [13]:
casos_abandono = n * p_gorro
casos_permanencia = n * (1 - p_gorro)
print("n por p gorro (clientes que abandonan):", round(casos_abandono))
print("n por (1 - p gorro) (clientes que se quedan):", round(casos_permanencia))
print("¿Ambos son al menos 10?:", casos_abandono >= 10 and casos_permanencia >= 10)

n por p gorro (clientes que abandonan): 206
n por (1 - p gorro) (clientes que se quedan): 794
¿Ambos son al menos 10?: True


---
## Paso 9 · Cómo se interpreta el intervalo

**Interpretación correcta (así se escribe en el reporte):**
> Con un nivel de confianza del 95 %, la proporción de clientes del banco que abandonan está entre **0.182 y 0.230** (18.2 % y 23.0 %), es decir, entre unos 1,820 y 2,300 de los 10,000 clientes.

**Qué significa el 95 %:** si se repitiera el muestreo muchas veces y con cada muestra se construyera un intervalo de esta forma, cerca del 95 % de esos intervalos contendrían la proporción real. El 95 % es la tasa de acierto **del método**.

**Error común que hay que evitar:** decir "hay 95 % de probabilidad de que p esté entre 0.182 y 0.230". La proporción real $p$ es un número fijo: está o no está dentro del intervalo. Lo que cambia de una muestra a otra es el intervalo.

**Comprobación posible sólo en nuestro proyecto:** como tenemos la población, sabemos que $p$ = 0.2037. Está dentro del intervalo (0.182, 0.230).

---
## Paso 10 · Otra fórmula posible: el intervalo de Wilson

Existe otra fórmula para el intervalo de una proporción, llamada **intervalo de Wilson**. Se calcula con scipy (sin corrección por población finita, para compararlo con el intervalo del Paso 7):

In [14]:
resultado_binomial = stats.binomtest(int(muestra["abandono"].sum()), n)
intervalo_wilson = resultado_binomial.proportion_ci(confidence_level=0.95, method="wilson")
print("Intervalo de Wald (Paso 7):", (float(round(limite_inferior_sin_correccion, 4)), float(round(limite_superior_sin_correccion, 4))))
print("Intervalo de Wilson:", (float(round(intervalo_wilson.low, 4)), float(round(intervalo_wilson.high, 4))))

Intervalo de Wald (Paso 7): (0.1809, 0.2311)
Intervalo de Wilson: (0.1821, 0.2322)


| Fórmula | A favor | En contra |
|---|---|---|
| **Wald** (la del documento, la que usa el proyecto) | Es la que pide el documento. Es fácil de calcular a mano y de explicar | Falla cuando la muestra es pequeña o la proporción está cerca de 0 o de 1 |
| **Wilson** | Funciona bien también con muestras pequeñas o proporciones extremas | Su fórmula es más complicada y no viene en el documento |

Con nuestra muestra (1,000 clientes y proporción de 0.206) los dos intervalos son casi iguales. Por eso el proyecto usa la fórmula de Wald, que es la del documento.

---
## Preguntas que puede hacer el profesor (apartado 4 y primera parte del 5)

**1. ¿Cuál es la diferencia entre parámetro y estadístico?**
El parámetro es el valor de toda la población y no se conoce. El estadístico es el valor calculado con la muestra y se usa para estimar el parámetro.

**2. Interpreten su estimación puntual de P1.**
La proporción muestral de abandono fue de 0.206. Es una estimación puntual de la proporción de abandono de los 10,000 clientes.

**3. ¿Por qué no basta con la estimación puntual?**
Porque otra muestra daría otro valor, y un solo número no dice cuánto se puede alejar del valor real. El intervalo sí lo dice.

**4. ¿Qué es el error estándar?**
Es cuánto varía la estimación de una muestra a otra. Para P1 vale 0.0128 (0.0121 con la corrección).

**5. ¿De dónde sale el 1.96?**
En una distribución normal, el 95 % de los valores está a menos de 1.96 errores estándar del centro. Es el valor crítico para 95 % de confianza.

**6. ¿Qué es el factor de corrección por población finita y por qué lo usaron?**
Es $\sqrt{(N-n)/(N-1)}$. Se usa cuando la muestra es más del 5 % de la población; la nuestra es el 10 %. Vale 0.9487 y reduce el error estándar un 5 %.

**7. Interpreten su intervalo.**
Con 95 % de confianza, la proporción de clientes que abandonan está entre 0.182 y 0.230.

**8. ¿Hay 95 % de probabilidad de que la proporción real esté en su intervalo?**
No. La proporción real es fija. El 95 % es la proporción de intervalos que contendrían el valor real si el muestreo se repitiera muchas veces.

**9. ¿Qué condición se necesita para usar esta fórmula?**
Muestra aleatoria y al menos 10 casos de cada resultado. Tenemos 206 que abandonan y 794 que se quedan.

**10. ¿Por qué no usaron el intervalo de Wilson?**
Porque el documento usa la fórmula de Wald y, con 1,000 clientes y una proporción de 0.206, las dos dan casi el mismo intervalo.

---
## Tarea del Día 3 (para todo el equipo)

Cada integrante calcula **a mano, con calculadora**, el intervalo de P1 siguiendo los pasos 5, 7 y 8: error estándar, margen de error, FCPF e intervalo. Comparen su resultado con el del cuaderno.

➡️ **Día 4:** los otros tres intervalos (P2, P3 y P4).